In [1]:
import pandas as pd
from tqdm import tqdm
import os

RAW_DATA_PATH = '../data/raw/twcs.csv'
PROCESSED_PATH = '../data/processed/clean_brand_threads.csv'
BRAND_NAME = 'AppleSupport'

# 1. Stream the CSV to find AppleSupport's replies
chunk_size = 100000
brand_replies_list = []

print(f"Scanning for {BRAND_NAME} replies...")
for chunk in tqdm(pd.read_csv(RAW_DATA_PATH, chunksize=chunk_size)):
    # Keep only inbound replies from AppleSupport
    brand_mask = (chunk['author_id'] == BRAND_NAME) & (chunk['in_response_to_tweet_id'].notna())
    brand_replies_list.append(chunk[brand_mask])

df_brand = pd.concat(brand_replies_list, ignore_index=True)
print(f"Found {len(df_brand)} replies from {BRAND_NAME}.")

# Sample 5,000 to keep the next step fast
df_brand_sample = df_brand.sample(5000, random_state=42)

# Clean the IDs (Pandas sometimes loads large IDs as floats like 1.23e18)
df_brand_sample['in_response_to_tweet_id'] = pd.to_numeric(df_brand_sample['in_response_to_tweet_id'], errors='coerce').astype('Int64').astype(str)
customer_tweet_ids = set(df_brand_sample['in_response_to_tweet_id'])

Scanning for AppleSupport replies...


29it [01:19,  2.75s/it]

Found 106719 replies from AppleSupport.


In [2]:
customer_tweets_list = []
print("Retrieving corresponding customer tweets...")

for chunk in tqdm(pd.read_csv(RAW_DATA_PATH, chunksize=chunk_size)):
    chunk['tweet_id'] = chunk['tweet_id'].astype(str)
    customer_mask = chunk['tweet_id'].isin(customer_tweet_ids)
    customer_tweets_list.append(chunk[customer_mask])

df_customer = pd.concat(customer_tweets_list, ignore_index=True)
df_customer['tweet_id'] = df_customer['tweet_id'].astype(str)
print(f"Found {len(df_customer)} original customer tweets.")

# Merge customer tweet and agent reply on the IDs
df_threads = pd.merge(
    df_customer, 
    df_brand_sample, 
    left_on='tweet_id', 
    right_on='in_response_to_tweet_id', 
    suffixes=('_customer', '_agent')
)

# Keep only what we need for the AI
df_clean = pd.DataFrame({
    'thread_id': df_threads['tweet_id_customer'],
    'customer_text': df_threads['text_customer'],
    'agent_text': df_threads['text_agent']
})

# Drop tiny tweets (like "thanks" or just emojis)
df_clean = df_clean[df_clean['customer_text'].str.len() > 15]

# Save exactly 2,000 threads
df_final = df_clean.sample(2000, random_state=42).reset_index(drop=True)
os.makedirs('../data/processed', exist_ok=True)
df_final.to_csv(PROCESSED_PATH, index=False)

print(f"Phase 1 Complete! Saved {len(df_final)} clean threads to {PROCESSED_PATH}.")
display(df_final.head())

Retrieving corresponding customer tweets...


29it [01:21,  2.81s/it]


Found 4995 original customer tweets.
Phase 1 Complete! Saved 2000 clean threads to ../data/processed/clean_brand_threads.csv.


,thread_id,customer_text,agent_text
0,2756810,@AppleSupport No 😢,@771545 Thank you. Do you share your Apple ID ...
1,1626283,"Woke up this am, reached in my nightstand and ...",@497991 Thanks for reaching out. Let's take a ...
2,2134870,@115858 literally is gonna get hit. Like i hav...,@628087 We'd like to get caught up on the issu...
3,889179,I’m not playing when I say this. I turned it o...,@148421 Let's move over to a DM where we can t...
4,1907373,@AppleSupport i changed my Apple ID on my phon...,@568294 Your music is important! Let us know w...
